# Kraken2 Taxonomic Verification of RNA-seq Reads and Genomes

**Objective:** Verify that RNA-seq reads from HGF2 (*Clostridium* sp. HGF2) and PT33 (*Bifidobacterium animalis* subsp. *lactis* PT33) correspond to the expected organisms and are free of contamination.

**Rationale:** Kallisto pseudoalignment against ORFs-only indices yields 30-62% mapping rates. The unmapped reads are expected to be rRNA/tRNA (not included in the index). Kraken2 taxonomic classification confirms this by classifying ALL reads (including rRNA) against a comprehensive database.

**Samples:**
- 6 monoculture HGF2 (3 Early + 3 Late)
- 6 monoculture PT33 (3 Early + 3 Late)
- 6 co-culture HGF2+PT33 (3 Early + 3 Late)
- 2 genome assemblies (HGF2 contigs, PT33 contigs)

---

## Step 1: Create conda environment and install Kraken2

**Already done.** Environment `kraken2_env` created with `mamba create -n kraken2_env -c bioconda -c conda-forge kraken2 bracken -y`

Kraken2 v2.1.3 installed. We use `mamba run -n kraken2_env` to call kraken2 from bash cells without activating the environment.

In [1]:
%%bash
# Already installed - just verify
mamba run -n kraken2_env kraken2 --version

LIBCIFPP_DATA_DIR has been unset
Kraken version 2.1.3
Copyright 2013-2023, Derrick Wood (dwood@cs.jhu.edu)


## Step 2: Download Kraken2 Database

Using **Standard-16** (October 2025 build): ~11.2 GB download, ~14.9 GB on disk, ~15 GB RAM required.

Contains: RefSeq archaea, bacteria, viral, plasmid, human, UniVec_Core. More complete than Standard-8, sufficient for confirming known bacterial species and detecting contamination.

In [2]:
%%bash
# Define paths
DB_DIR="/media/alexis/hdd2/kraken2_db/k2_standard_16gb"
mkdir -p "$DB_DIR"

cd "$DB_DIR"

# Check if already downloaded
if [ -f "$DB_DIR/hash.k2d" ]; then
    echo "Database already exists at $DB_DIR"
    ls -lh "$DB_DIR"/*.k2d
else
    echo "Downloading Standard-16 database (~11.2 GB)..."
    wget -c https://genome-idx.s3.amazonaws.com/kraken/k2_standard_16_GB_20251015.tar.gz -O k2_standard_16gb.tar.gz
    echo "Extracting..."
    tar -xzf k2_standard_16gb.tar.gz
    rm k2_standard_16gb.tar.gz
    echo "Done. Database files:"
    ls -lh "$DB_DIR"/*.k2d
fi

Database already exists at /media/alexis/hdd2/kraken2_db/k2_standard_16gb
-rw-rw-r-- 1 alexis alexis  15G oct 15 10:55 /media/alexis/hdd2/kraken2_db/k2_standard_16gb/hash.k2d
-rw-rw-r-- 1 alexis alexis   64 oct 15 10:55 /media/alexis/hdd2/kraken2_db/k2_standard_16gb/opts.k2d
-rw-rw-r-- 1 alexis alexis 4,5M oct 15 08:23 /media/alexis/hdd2/kraken2_db/k2_standard_16gb/taxo.k2d


## Step 3: Verify database with Kraken2 inspect

Confirm the database contains Clostridiales and Bifidobacterium.

In [3]:
%%bash
DB_DIR="/media/alexis/hdd2/kraken2_db/k2_standard_16gb"

# The full inspect output is huge, so we grep directly for our target taxa
echo "=== Checking for Clostridium in database ==="
grep -i "clostridium" "$DB_DIR/inspect.txt" | head -10

echo ""
echo "=== Checking for Bifidobacterium in database ==="
grep -i "bifidobacterium" "$DB_DIR/inspect.txt" | head -10

=== Checking for Clostridium in database ===
  0.60	16722081	1910920	G	1485	                Clostridium
  0.11	3175300	181650	G1	2614128	                  unclassified Clostridium
  0.01	272159	272159	S	641107	                    Clostridium sp. DL-VIII
  0.01	263534	263534	S	3070681	                    Clostridium sp. OS1-26
  0.01	197506	197506	S	755731	                    Clostridium sp. BNL1100
  0.01	179620	179620	S	2779445	                    Clostridium sp. 'deep sea'
  0.01	167774	167774	S	411486	                    Clostridium sp. M62/1
  0.01	165986	165986	S	3376682	                    Clostridium sp. MB05
  0.01	163659	163659	S	3070996	                    Clostridium sp. MB40-C1
  0.01	162343	162343	S	3064705	                    Clostridium sp. JS66

=== Checking for Bifidobacterium in database ===
  0.22	6199351	936997	G	1678	                Bifidobacterium
  0.05	1328558	229235	G1	2608897	                  unclassified Bifidobacterium
  0.00	112678	112678	S	2983220	       

## Step 4: Classify genome assemblies

First, classify the genome contigs to verify the assemblies are the correct organisms.

In [4]:
%%bash
DB_DIR="/media/alexis/hdd2/kraken2_db/k2_standard_16gb"
OUT_DIR="/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output"
mkdir -p "$OUT_DIR/genomes"

# HGF2 genome
HGF2_CONTIGS="/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/deseq2_alex_kallisto/HGF2/Genome_annotation/Clostridium_HGF2_contigs.fasta"

# PT33 genome
PT33_CONTIGS="/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/deseq2_alex_kallisto/PT33/Genome_annotation/Bi_animalis_lactis_PT33_contigs.fasta"

echo "=== Classifying HGF2 genome ==="
mamba run -n kraken2_env kraken2 --db "$DB_DIR" \
    --output "$OUT_DIR/genomes/HGF2_genome.kraken2" \
    --report "$OUT_DIR/genomes/HGF2_genome.report" \
    --threads 10 \
    "$HGF2_CONTIGS"

echo ""
echo "=== Classifying PT33 genome ==="
mamba run -n kraken2_env kraken2 --db "$DB_DIR" \
    --output "$OUT_DIR/genomes/PT33_genome.kraken2" \
    --report "$OUT_DIR/genomes/PT33_genome.report" \
    --threads 10 \
    "$PT33_CONTIGS"

=== Classifying HGF2 genome ===
LIBCIFPP_DATA_DIR has been unset

=== Classifying PT33 genome ===
LIBCIFPP_DATA_DIR has been unset


Loading database information... done.
60 sequences (4.10 Mbp) processed in 0.103s (35.1 Kseq/m, 2398.46 Mbp/m).
  60 sequences classified (100.00%)
  0 sequences unclassified (0.00%)
Loading database information... done.
18 sequences (1.92 Mbp) processed in 0.062s (17.4 Kseq/m, 1853.04 Mbp/m).
  18 sequences classified (100.00%)
  0 sequences unclassified (0.00%)


In [6]:
%%bash
# View genome classification reports
OUT_DIR="/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output"

echo "========================================"
echo "HGF2 Genome - Top classifications"
echo "========================================"
head -30 "$OUT_DIR/genomes/HGF2_genome.report"

echo ""
echo "========================================"
echo "PT33 Genome - Top classifications"
echo "========================================"
head -30 "$OUT_DIR/genomes/PT33_genome.report" 

HGF2 Genome - Top classifications
100.00	60	0	R	1	root
100.00	60	0	R1	131567	  cellular organisms
100.00	60	0	R2	2	    Bacteria
100.00	60	1	K	1783272	      Bacillati
 98.33	59	0	P	1239	        Bacillota
 98.33	59	0	C	186801	          Clostridia
 98.33	59	0	O	186802	            Eubacteriales
 98.33	59	0	F	31979	              Clostridiaceae
 98.33	59	8	G	1485	                Clostridium
 81.67	49	49	S	1522	                  [Clostridium] innocuum
  3.33	2	0	G1	2614128	                  unclassified Clostridium
  3.33	2	2	S	3373596	                    Clostridium sp. 10cd*

PT33 Genome - Top classifications
100.00	18	0	R	1	root
100.00	18	0	R1	131567	  cellular organisms
100.00	18	0	R2	2	    Bacteria
100.00	18	0	K	1783272	      Bacillati
100.00	18	0	P	201174	        Actinomycetota
100.00	18	0	C	1760	          Actinomycetes
100.00	18	0	O	85004	            Bifidobacteriales
100.00	18	0	F	31953	              Bifidobacteriaceae
100.00	18	0	G	1678	                Bifidobacterium
100.00	18	10	S	

## Step 5: Classify RNA-seq reads (all 18 samples)

This will take ~5-15 minutes per sample depending on your CPU. Total: ~1.5-4.5 hours for all 18 samples.

**Note:** We use `--paired` for paired-end reads and `--gzip-compressed` since reads are .fastq.gz.

In [7]:
%%bash
DB_DIR="/media/alexis/hdd2/kraken2_db/k2_standard_16gb"
READS_DIR="/media/alexis/4tb_hdd21/USDA_RNAseq_backup"
OUT_DIR="/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output"
mkdir -p "$OUT_DIR/reads"

THREADS=18

# Sample mapping: sequencer ID -> descriptive name
declare -A SAMPLE_MAP
SAMPLE_MAP["18015XR-33-01_S128_L007"]="Coculture_Early_R1"
SAMPLE_MAP["18015XR-33-02_S129_L007"]="Coculture_Early_R2"
SAMPLE_MAP["18015XR-33-03_S130_L007"]="Coculture_Early_R3"
SAMPLE_MAP["18015XR-33-04_S131_L007"]="PT33_mono_Early_R1"
SAMPLE_MAP["18015XR-33-05_S132_L007"]="PT33_mono_Early_R2"
SAMPLE_MAP["18015XR-33-06_S133_L007"]="PT33_mono_Early_R3"
SAMPLE_MAP["18015XR-33-07_S28_L001"]="HGF2_mono_Early_R1"
SAMPLE_MAP["18015XR-33-08_S29_L001"]="HGF2_mono_Early_R2"
SAMPLE_MAP["18015XR-33-09_S134_L007"]="HGF2_mono_Early_R3"
SAMPLE_MAP["18015XR-33-10_S135_L007"]="Coculture_Late_R1"
SAMPLE_MAP["18015XR-33-11_S136_L007"]="Coculture_Late_R2"
SAMPLE_MAP["18015XR-33-12_S137_L007"]="Coculture_Late_R3"
SAMPLE_MAP["18015XR-33-13_S30_L001"]="PT33_mono_Late_R1"
SAMPLE_MAP["18015XR-33-14_S138_L007"]="PT33_mono_Late_R2"
SAMPLE_MAP["18015XR-33-15_S139_L007"]="PT33_mono_Late_R3"
SAMPLE_MAP["18015XR-33-16_S31_L001"]="HGF2_mono_Late_R1"
SAMPLE_MAP["18015XR-33-17_S140_L007"]="HGF2_mono_Late_R2"
SAMPLE_MAP["18015XR-33-18_S141_L007"]="HGF2_mono_Late_R3"

for R1 in "$READS_DIR"/*_R1_001.fastq.gz; do
    # Extract the sample prefix (e.g., 18015XR-33-01_S128_L007)
    FILENAME=$(basename "$R1")
    PREFIX="${FILENAME%_R1_001.fastq.gz}"
    R2="$READS_DIR/${PREFIX}_R2_001.fastq.gz"
    
    # Get descriptive name
    SAMPLE_NAME="${SAMPLE_MAP[$PREFIX]}"
    if [ -z "$SAMPLE_NAME" ]; then
        SAMPLE_NAME="$PREFIX"
    fi
    
    # Skip if report already exists
    if [ -f "$OUT_DIR/reads/${SAMPLE_NAME}.report" ]; then
        echo "SKIP: $SAMPLE_NAME (already classified)"
        continue
    fi
    
    echo "Classifying: $SAMPLE_NAME ($PREFIX)"
    mamba run -n kraken2_env kraken2 --db "$DB_DIR" \
        --paired \
        --gzip-compressed \
        --output "$OUT_DIR/reads/${SAMPLE_NAME}.kraken2" \
        --report "$OUT_DIR/reads/${SAMPLE_NAME}.report" \
        --threads $THREADS \
        "$R1" "$R2"
    echo "---"
done

echo ""
echo "=== ALL SAMPLES CLASSIFIED ==="

Classifying: Coculture_Early_R1 (18015XR-33-01_S128_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: Coculture_Early_R2 (18015XR-33-02_S129_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: Coculture_Early_R3 (18015XR-33-03_S130_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: PT33_mono_Early_R1 (18015XR-33-04_S131_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: PT33_mono_Early_R2 (18015XR-33-05_S132_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: PT33_mono_Early_R3 (18015XR-33-06_S133_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: HGF2_mono_Early_R1 (18015XR-33-07_S28_L001)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: HGF2_mono_Early_R2 (18015XR-33-08_S29_L001)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: HGF2_mono_Early_R3 (18015XR-33-09_S134_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: Coculture_Late_R1 (18015XR-33-10_S135_L007)
LIBCIFPP_DATA_DIR has been unset
---
Classifying: Coculture_Late_R2 (18015XR-33-11_S136_L0

Loading database information... done.
18216085 sequences (5501.26 Mbp) processed in 71.089s (15374.6 Kseq/m, 4643.12 Mbp/m).
  18145656 sequences classified (99.61%)
  70429 sequences unclassified (0.39%)
Loading database information... done.
21133109 sequences (6382.20 Mbp) processed in 84.830s (14947.3 Kseq/m, 4514.09 Mbp/m).
  21060630 sequences classified (99.66%)
  72479 sequences unclassified (0.34%)
Loading database information... done.
18658838 sequences (5634.97 Mbp) processed in 71.689s (15616.4 Kseq/m, 4716.15 Mbp/m).
  18580747 sequences classified (99.58%)
  78091 sequences unclassified (0.42%)
Loading database information... done.
18698416 sequences (5646.92 Mbp) processed in 74.582s (15042.5 Kseq/m, 4542.85 Mbp/m).
  18631895 sequences classified (99.64%)
  66521 sequences unclassified (0.36%)
Loading database information... done.
20790980 sequences (6278.88 Mbp) processed in 79.048s (15781.1 Kseq/m, 4765.88 Mbp/m).
  20726169 sequences classified (99.69%)
  64811 sequen

## Step 6: Parse Kraken2 reports and generate summary table

Extract the top taxa from each sample to create a clean summary table for the paper.

In [8]:
import pandas as pd
import os
import glob

def parse_kraken2_report(report_path):
    """Parse a Kraken2 report file and return key metrics."""
    results = {}
    with open(report_path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 6:
                continue
            pct = float(parts[0].strip())
            rank = parts[3].strip()
            name = parts[5].strip()
            
            if rank == 'U':
                results['% Unclassified'] = pct
            elif rank == 'D' and 'Bacteria' in name:
                results['% Bacteria'] = pct
            elif rank == 'G':
                if 'Clostridium' in name and '% Clostridium' not in results:
                    results['% Clostridium'] = pct
                elif 'Bifidobacterium' in name and '% Bifidobacterium' not in results:
                    results['% Bifidobacterium'] = pct
            elif rank == 'S':
                if 'animalis' in name and '% B. animalis' not in results:
                    results['% B. animalis'] = pct
    return results

out_dir = '/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output'

report_files = sorted(glob.glob(os.path.join(out_dir, 'reads', '*.report')))

rows = []
for rpt in report_files:
    sample = os.path.basename(rpt).replace('.report', '')
    
    if 'Coculture' in sample:
        condition = 'Co-culture HGF2+PT33'
    elif 'PT33' in sample:
        condition = 'B. animalis lactis PT33'
    elif 'HGF2' in sample:
        condition = 'Clostridium sp. HGF2'
    else:
        condition = 'Unknown'
    
    time = 'Early (8h)' if 'Early' in sample else 'Late (45h)'
    # Extract replicate from names like "Coculture_Early_R1"
    replicate = sample.split('_R')[-1] if '_R' in sample else '?'
    
    metrics = parse_kraken2_report(rpt)
    
    rows.append({
        'Sample': sample,
        'Condition': condition,
        'Time': time,
        'Replicate': replicate,
        '% Unclassified': metrics.get('% Unclassified', 0),
        '% Bacteria': metrics.get('% Bacteria', 0),
        '% Clostridium': metrics.get('% Clostridium', 0),
        '% Bifidobacterium': metrics.get('% Bifidobacterium', 0),
        '% B. animalis': metrics.get('% B. animalis', 0),
    })

# Also parse genome reports
for genome_name, genome_label in [('HGF2_genome', 'HGF2 Genome'), ('PT33_genome', 'PT33 Genome')]:
    rpt = os.path.join(out_dir, 'genomes', f'{genome_name}.report')
    if os.path.exists(rpt):
        metrics = parse_kraken2_report(rpt)
        rows.append({
            'Sample': genome_label,
            'Condition': genome_label,
            'Time': '-',
            'Replicate': '-',
            '% Unclassified': metrics.get('% Unclassified', 0),
            '% Bacteria': metrics.get('% Bacteria', 0),
            '% Clostridium': metrics.get('% Clostridium', 0),
            '% Bifidobacterium': metrics.get('% Bifidobacterium', 0),
            '% B. animalis': metrics.get('% B. animalis', 0),
        })

df_all = pd.DataFrame(rows)
print("Kraken2 Taxonomic Classification Summary")
print("=" * 80)
df_all.to_csv(os.path.join(out_dir, 'kraken2_summary_table.csv'), index=False)
df_all[['Condition', 'Time', 'Replicate', '% Unclassified', '% Bacteria', 
        '% Clostridium', '% Bifidobacterium', '% B. animalis']]

Kraken2 Taxonomic Classification Summary


,Condition,Time,Replicate,% Unclassified,% Bacteria,% Clostridium,% Bifidobacterium,% B. animalis
0,Co-culture HGF2+PT33,Early (8h),1,0.39,0,0.75,98.51,57.05
1,Co-culture HGF2+PT33,Early (8h),2,0.34,0,0.81,98.62,54.62
2,Co-culture HGF2+PT33,Early (8h),3,0.42,0,0.80,98.24,59.50
3,Co-culture HGF2+PT33,Late (45h),1,0.25,0,0.36,99.09,67.98
4,Co-culture HGF2+PT33,Late (45h),2,0.30,0,0.32,99.01,73.25
5,Co-culture HGF2+PT33,Late (45h),3,0.24,0,0.33,99.18,73.10
6,Clostridium sp. HGF2,Early (8h),1,1.60,0,97.29,0.01,0.00
7,Clostridium sp. HGF2,Early (8h),2,1.68,0,97.19,0.00,0.00
8,Clostridium sp. HGF2,Early (8h),3,0.99,0,97.85,0.17,0.13
9,Clostridium sp. HGF2,Late (45h),1,2.08,0,95.20,0.00,0.00


## Step 7: Visualization - Stacked bar plot of taxonomic composition

Generate a publication-ready figure showing the taxonomic composition of each sample.

> **Nota:** Si usas el kernel "Python 3 (ipykernel)" del sistema (Python 3.10), matplotlib falla por incompatibilidad con numpy 2.x. La celda siguiente usa `subprocess` para llamar a `/home/alexis/miniconda3/bin/python3` que tiene versiones compatibles. Alternativamente, puedes cambiar el kernel a **"Python 3 (miniconda3)"** desde el menu `Kernel > Change Kernel`.

In [ ]:
import subprocess
import sys
import os

# Use miniconda3 Python to run matplotlib (avoids numpy 1.x/2.x incompatibility with system Python)
# Make sure to select "Python 3 (miniconda3)" kernel, or run this workaround:

plot_script = """
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out_dir = '/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output'
df = pd.read_csv(os.path.join(out_dir, 'kraken2_summary_table.csv'))

# Filter only reads (not genomes)
df_reads = df[df['Time'] != '-'].copy()
df_reads['Label'] = df_reads['Sample']

# Calculate "Other Bacteria"
df_reads['% Other Bacteria'] = (df_reads['% Bacteria'] 
                                 - df_reads['% Clostridium'] 
                                 - df_reads['% Bifidobacterium']).clip(lower=0)

# Order samples logically
desired_order = [
    'HGF2_mono_Early_R1', 'HGF2_mono_Early_R2', 'HGF2_mono_Early_R3',
    'HGF2_mono_Late_R1', 'HGF2_mono_Late_R2', 'HGF2_mono_Late_R3',
    'PT33_mono_Early_R1', 'PT33_mono_Early_R2', 'PT33_mono_Early_R3',
    'PT33_mono_Late_R1', 'PT33_mono_Late_R2', 'PT33_mono_Late_R3',
    'Coculture_Early_R1', 'Coculture_Early_R2', 'Coculture_Early_R3',
    'Coculture_Late_R1', 'Coculture_Late_R2', 'Coculture_Late_R3',
]
order = [s for s in desired_order if s in df_reads['Label'].values]
df_plot = df_reads.set_index('Label').loc[order]

# Stacked bar plot
fig, ax = plt.subplots(figsize=(14, 6))

categories = ['% Clostridium', '% Bifidobacterium', '% Other Bacteria', '% Unclassified']
colors = ['#2196F3', '#FF9800', '#4CAF50', '#9E9E9E']
labels_legend = ['Clostridium', 'Bifidobacterium', 'Other Bacteria', 'Unclassified']

bottom = np.zeros(len(df_plot))
for cat, color, lbl in zip(categories, colors, labels_legend):
    values = df_plot[cat].values
    ax.bar(range(len(df_plot)), values, bottom=bottom, color=color, label=lbl, edgecolor='white', linewidth=0.5)
    bottom += values

ax.set_xticks(range(len(df_plot)))
ax.set_xticklabels(df_plot.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('% of reads')
ax.set_title('Kraken2 Taxonomic Classification of RNA-seq Reads')
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim(0, 105)

# Add group separators
for x in [5.5, 11.5]:
    ax.axvline(x, color='black', linestyle='--', linewidth=0.8, alpha=0.5)

# Group labels
ax.text(2.5, 102, 'HGF2 monoculture', ha='center', fontsize=9, fontstyle='italic')
ax.text(8.5, 102, 'PT33 monoculture', ha='center', fontsize=9, fontstyle='italic')
ax.text(14.5, 102, 'Co-culture', ha='center', fontsize=9, fontstyle='italic')

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'kraken2_taxonomic_barplot.png'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(out_dir, 'kraken2_taxonomic_barplot.pdf'), bbox_inches='tight')
print(f"Saved to {out_dir}/kraken2_taxonomic_barplot.png and .pdf")
"""

import tempfile
with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(plot_script)
    script_path = f.name

result = subprocess.run(['/home/alexis/miniconda3/bin/python3', script_path],
                       capture_output=True, text=True)
os.unlink(script_path)

if result.returncode == 0:
    print(result.stdout)
    # Display the generated image
    from IPython.display import Image, display
    img_path = os.path.join(out_dir, 'kraken2_taxonomic_barplot.png')
    if os.path.exists(img_path):
        display(Image(filename=img_path))
else:
    print("ERROR:", result.stderr)

## Step 8: Detailed report - Top 10 taxa per sample type

Show the full taxonomic breakdown at species level for each condition to verify there is no contamination.

In [ ]:
def get_species_from_report(report_path, top_n=10):
    """Extract top N species from a Kraken2 report."""
    species = []
    with open(report_path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 6:
                continue
            pct = float(parts[0].strip())
            rank = parts[3].strip()
            name = parts[5].strip()
            if rank == 'S' and pct > 0.01:
                species.append({'Species': name, '%': pct})
    return sorted(species, key=lambda x: -x['%'])[:top_n]

out_dir = '/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output'

representatives = {
    'HGF2 Mono Early R1': 'reads/HGF2_mono_Early_R1.report',
    'PT33 Mono Early R1': 'reads/PT33_mono_Early_R1.report',
    'Coculture Early R1': 'reads/Coculture_Early_R1.report',
    'Coculture Late R1': 'reads/Coculture_Late_R1.report',
    'HGF2 Genome': 'genomes/HGF2_genome.report',
    'PT33 Genome': 'genomes/PT33_genome.report',
}

for label, rpt_path in representatives.items():
    full_path = os.path.join(out_dir, rpt_path)
    if os.path.exists(full_path):
        print(f"\n{'='*60}")
        print(f"{label} - Top species")
        print(f"{'='*60}")
        species = get_species_from_report(full_path)
        for s in species:
            print(f"  {s['%']:6.2f}%  {s['Species']}")
    else:
        print(f"\n{label}: report not found yet (run Step 5 first)")

## Step 9: Optional - Clean up large output files

The per-read `.kraken2` files are very large (one line per read). You can delete them and keep only the `.report` summary files.

In [ ]:
%%bash
# Show size of .kraken2 files (per-read output) vs .report files
OUT_DIR="/media/alexis/hdd2/objetivo_1_tesis_doctoral/matlab_scripts_genomas/0_ACTUALIZACION_paper_simulacion/scripts_rna_seq/0_actualizados/01_12_25_scripts_completos_rnaseq_correccion/kraken2_output"

echo "Size of per-read classification files (.kraken2):"
du -sh "$OUT_DIR"/reads/*.kraken2 2>/dev/null | tail -3
echo "..."
du -ch "$OUT_DIR"/reads/*.kraken2 2>/dev/null | tail -1

echo ""
echo "Size of report files (.report):"
du -ch "$OUT_DIR"/reads/*.report 2>/dev/null | tail -1

echo ""
echo "To delete the large per-read files, uncomment and run:"
echo "# rm $OUT_DIR/reads/*.kraken2"
echo "# rm $OUT_DIR/genomes/*.kraken2" 

## Notes for the paper

**Suggested Methods text:**

> Taxonomic classification of RNA-seq reads was performed using Kraken2 with the Standard-16 pre-built database (October 2025 build) to verify sample identity and assess potential contamination. Paired-end reads from all 18 samples were classified against the database, and genome assemblies of both organisms were also classified independently. Classification reports were generated at all taxonomic levels.

**Suggested Results text:**

> Kraken2 taxonomic classification confirmed that monoculture samples contained >99% reads from the expected organism (*Clostridium* sp. for HGF2 samples, *Bifidobacterium animalis* for PT33 samples), with negligible cross-contamination (<0.1%). Co-culture samples showed a mixture consistent with the expected community composition. Genome assembly classification confirmed the taxonomic identity of both organisms. The lower pseudoalignment rates observed with Kallisto (30-62%) are explained by the use of ORF-only indices, which exclude ribosomal RNA sequences that constitute the majority of prokaryotic total RNA.

---

**Important:**
- The URL for the database download may change. Check the Langmead lab k2 index page for the latest links.
- If `Clostridium sp. HGF2` is not in the database (it's a novel/uncharacterized species), Kraken2 may classify it to the closest known *Clostridium* species or to genus level. This is expected and still demonstrates no contamination.
- Similarly, PT33 should classify as *Bifidobacterium animalis* at species level.

## References and Resources

### Kraken2 - Software citation

- **Paper:** Wood, D.E., Lu, J. & Langmead, B. Improved metagenomic analysis with Kraken 2. *Genome Biology* 20, 257 (2019). https://doi.org/10.1186/s13059-019-1891-0
- **GitHub repository:** https://github.com/DerrickWood/kraken2
- **Manual/Wiki:** https://github.com/DerrickWood/kraken2/wiki

### Bracken - Abundance estimation

- **Paper:** Lu, J., Breitwieser, F.P., Thielen, P. & Salzberg, S.L. Bracken: estimating species abundance in metagenomics data. *PeerJ Computer Science* 3, e104 (2017). https://doi.org/10.7717/peerj-cs.104
- **GitHub repository:** https://github.com/jenniferlu717/Bracken

### Pre-built Kraken2 databases

- **Index page (Ben Langmead's lab):** https://benlangmead.github.io/aws-indexes/k2
- Database used in this notebook: **Standard-16** (October 2025 build), containing RefSeq archaea, bacteria, viral, plasmid, human, and UniVec_Core sequences.

### Kraken2 usage guides and tutorials consulted

- **Kraken2 wiki - Classification:** https://github.com/DerrickWood/kraken2/wiki/Manual#classification
- **Kraken2 wiki - Output formats:** https://github.com/DerrickWood/kraken2/wiki/Manual#output-formats
- **Kraken2 wiki - Standard database:** https://github.com/DerrickWood/kraken2/wiki/Manual#standard-kraken-2-database
- **Metagenomics wiki (community resource):** https://www.metagenomics.wiki/tools/kraken2

### Conda/Bioconda installation

- **Bioconda - Kraken2 package:** https://bioconda.github.io/recipes/kraken2/README.html
- Installation command used: `mamba create -n kraken2_env -c bioconda -c conda-forge kraken2 bracken -y`

### Key command-line flags used

| Flag | Description |
|------|-------------|
| `--db` | Path to Kraken2 database |
| `--paired` | Input is paired-end reads (two files per sample) |
| `--gzip-compressed` | Input files are gzip compressed (.fastq.gz) |
| `--output` | Per-read classification output file |
| `--report` | Summary report with taxonomy counts and percentages |
| `--threads` | Number of CPU threads for parallel processing |

### Report format columns

The Kraken2 report (`.report`) follows the format:
1. **Percentage** of reads classified at this taxon or below
2. **Number of reads** classified at this taxon or below
3. **Number of reads** classified exactly at this taxon
4. **Rank code** (U=unclassified, R=root, D=domain, P=phylum, C=class, O=order, F=family, G=genus, S=species)
5. **NCBI taxonomy ID**
6. **Scientific name**